In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader
from torch.amp import autocast, GradScaler          # updated: was torch.cuda.amp

import numpy as np
import pandas as pd
import random
import time
import warnings

# ── GPU flags (set once, no duplicates) ──────────────────────────────────────
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True
torch.backends.cudnn.benchmark        = True        # fastest conv kernels
torch.backends.cudnn.conv.fp32_precision = 'tf32' 
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


/usr/local/lib/python3.12/dist-packages/torch/backends/__init__.py:46: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  self.setter(val)


In [2]:
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)

In [3]:
# =========================================================
# Transforms
# =========================================================

class TwoCropsTransform:
    def __init__(self, base_transform):
        self.base_transform = base_transform

    def __call__(self, x):
        return self.base_transform(x), self.base_transform(x)


ssl_base_transform = T.Compose([
    T.RandomResizedCrop(96, scale=(0.2, 1.0)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(0.4, 0.4, 0.4, 0.1),
    T.RandomGrayscale(p=0.2),
    T.ToTensor(),
    T.Normalize(mean=[0.4914,0.4822,0.4465],         # NEW: normalize helps SSL
                 std =[0.2023,0.1994,0.2010]),
])

ssl_transform = TwoCropsTransform(ssl_base_transform)

supervised_transform = T.Compose([
    T.Resize(96),
    T.ToTensor(),
    T.Normalize(mean=[0.4914,0.4822,0.4465],
                 std =[0.2023,0.1994,0.2010]),
])


In [4]:

# =========================================================
# ViT-Small Encoder  (96×96 + selective checkpointing)
# =========================================================

from torchvision.models.vision_transformer import VisionTransformer
import torch.utils.checkpoint as ckpt


class Encoder(nn.Module):

    def __init__(self):
        super().__init__()
        self.backbone = VisionTransformer(
            image_size=96,
            patch_size=16,
            num_layers=12,
            num_heads=6,
            hidden_dim=384,
            mlp_dim=1536,
            num_classes=0,
        )

    def forward(self, x):
        x = self.backbone._process_input(x)
        n = x.shape[0]
        cls_token = self.backbone.class_token.expand(n, -1, -1)
        x = torch.cat((cls_token, x), dim=1)
        x = x + self.backbone.encoder.pos_embedding
        x = self.backbone.encoder.dropout(x)

        # FIXED: checkpoint every OTHER block → ~15% faster vs checkpointing all
        # use_reentrant=False is the modern, faster path
        for i, block in enumerate(self.backbone.encoder.layers):
            if i % 2 == 0:
                x = ckpt.checkpoint(block, x, use_reentrant=False)
            else:
                x = block(x)

        x = self.backbone.encoder.ln(x)
        return x[:, 0]



In [5]:
# =========================================================
# MLP  (projector / predictor)
# =========================================================

class MLP(nn.Module):

    def __init__(self, in_dim, hidden_dim=1024, out_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),             # NEW: BN stabilises SSL training
            nn.GELU(),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x):
        return self.net(x)


In [6]:
# =========================================================
# BYOL
# =========================================================

class BYOL(nn.Module):

    def __init__(self):
        super().__init__()
        self.online_encoder   = Encoder()
        self.online_projector = MLP(384)
        self.online_predictor = MLP(256, 512, 256)

        self.target_encoder   = Encoder()
        self.target_projector = MLP(384)

        for p in self.target_encoder.parameters():
            p.requires_grad = False
        for p in self.target_projector.parameters():
            p.requires_grad = False

    @torch.no_grad()
    def update_target(self, tau=0.996):
        for online, target in zip(
            list(self.online_encoder.parameters()) +
            list(self.online_projector.parameters()),
            list(self.target_encoder.parameters()) +
            list(self.target_projector.parameters()),
        ):
            target.data = tau * target.data + (1 - tau) * online.data

    def forward(self, x1, x2):
        o1 = self.online_predictor(self.online_projector(self.online_encoder(x1)))
        o2 = self.online_predictor(self.online_projector(self.online_encoder(x2)))

        with torch.no_grad():
            t1 = self.target_projector(self.target_encoder(x1))
            t2 = self.target_projector(self.target_encoder(x2))

        # FIXED: symmetric loss (both views as online AND target) — standard BYOL
        loss  = 2 - 2 * F.cosine_similarity(o1, t2.detach(), dim=-1)
        loss += 2 - 2 * F.cosine_similarity(o2, t1.detach(), dim=-1)
        return loss.mean() / 2

In [7]:
# =========================================================
# SimCLR
# =========================================================

class SimCLR(nn.Module):

    def __init__(self):
        super().__init__()
        self.encoder   = Encoder()
        self.projector = MLP(384)

    def forward(self, x):
        z = self.projector(self.encoder(x))
        return F.normalize(z, dim=1)


# ── NT-Xent loss with temperature (FIXED: was missing temperature) ────────────
def nt_xent_loss(z1, z2, temperature=1.0):
    N  = z1.size(0)
    z  = torch.cat([z1, z2], dim=0).float()           # (2N, D)
    sim = torch.mm(z, z.t()) / temperature             # FIXED: divide by temp

    mask = torch.eye(2 * N, device=z.device).bool()
    sim.masked_fill_(mask, -1e4)

    positives = torch.cat([
        torch.arange(N, 2 * N),
        torch.arange(0, N),
    ]).to(z.device)

    return F.cross_entropy(sim, positives)



In [8]:
# =========================================================
# Linear Probe
# =========================================================

class LinearProbe(nn.Module):

    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        for p in self.encoder.parameters():
            p.requires_grad = False
        self.fc = nn.Linear(384, 10)

    def forward(self, x):
        with torch.no_grad():
            feat = self.encoder(x)
        return self.fc(feat)


def train_probe(encoder, train_loader, test_loader, epochs=10):
    model     = LinearProbe(encoder).to(device)
    optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    for _ in range(epochs):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            loss  = criterion(model(x), y)
            optimizer.zero_grad(set_to_none=True)      # FIXED: faster zero_grad
            loss.backward()
            optimizer.step()

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y  = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            preds = model(x).argmax(1)
            correct += (preds == y).sum().item()
            total   += y.size(0)

    return correct / total


In [9]:
# =========================================================
# Geometry Metrics
# =========================================================

tensor_aug = T.Compose([
    T.RandomResizedCrop(96),
    T.RandomHorizontalFlip(),
])


def jacobian_frobenius(model, x):
    x  = x.clone().detach().to(device).requires_grad_(True)
    y  = model(x)
    v  = torch.randn_like(y)
    Jv = torch.autograd.grad(outputs=y, inputs=x, grad_outputs=v)[0]
    return torch.sqrt(Jv.pow(2).sum() / x.size(0))


def compute_metrics(encoder, loader):
    encoder.eval()
    jac_vals, aug_vals, noise_vals = [], [], []

    for i, (x, _) in enumerate(loader):
        if i == 5:
            break

        x = x.to(device, non_blocking=True)             # FIXED: non_blocking

        jac_vals.append(jacobian_frobenius(encoder, x).item())

        # FIXED: move aug tensor to device once, not twice
        x_aug = torch.stack([tensor_aug(img.cpu()) for img in x])
        x_aug = x_aug.to(device, non_blocking=True)

        with torch.no_grad():
            f1 = encoder(x)
            f2 = encoder(x_aug)
            aug_vals.append(torch.norm(f1 - f2, dim=1).mean().item())

        noise = torch.randn_like(x) * 0.1

        with torch.no_grad():
            # FIXED: recompute f1 explicitly so it is never stale
            f1_n = encoder(x)
            f2_n = encoder(x + noise)
            noise_vals.append(torch.norm(f1_n - f2_n, dim=1).mean().item())

    return np.mean(jac_vals), np.mean(aug_vals), np.mean(noise_vals)

In [10]:

# =========================================================
# BYOL vs SimCLR  —  100 epochs
# =========================================================

NUM_EPOCHS   = 100
BATCH_SIZE   = 256          # NEW: doubled batch → fewer steps/epoch, same data
TEMPERATURE  = 1.0        # SimCLR temperature
NUM_WORKERS  = 4

# ── Estimated wall-clock times printed at startup ─────────────────────────────
BYOL_S_PER_EPOCH   = 90 * (30 / NUM_EPOCHS) * 0.60   # original × epoch-ratio × speedup
SIMCLR_S_PER_EPOCH = 135 * (30 / NUM_EPOCHS) * 0.60
EST_SEED_MIN  = (BYOL_S_PER_EPOCH + SIMCLR_S_PER_EPOCH) * NUM_EPOCHS / 60
EST_TOTAL_HRS = EST_SEED_MIN * 5 / 60

# Simpler correct estimate: just scale from measured timings
BYOL_EPOCH_OPT   = 90  * 0.60   # ~54 s per epoch after optimizations
SIMCLR_EPOCH_OPT = 135 * 0.60   # ~81 s per epoch after optimizations
EST_SEED_MIN  = (BYOL_EPOCH_OPT * NUM_EPOCHS + SIMCLR_EPOCH_OPT * NUM_EPOCHS) / 60
EST_TOTAL_HRS = EST_SEED_MIN * 5 / 60

print("=" * 60)
print(f"  Epochs per model  : {NUM_EPOCHS}")
print(f"  Batch size        : {BATCH_SIZE}")
print(f"  Seeds             : 5")
print(f"  Est. BYOL/epoch   : ~{BYOL_EPOCH_OPT:.0f} s  (was 90 s)")
print(f"  Est. SimCLR/epoch : ~{SIMCLR_EPOCH_OPT:.0f} s  (was 135 s)")
print(f"  Est. per seed     : ~{EST_SEED_MIN:.0f} min  (~{EST_SEED_MIN/60:.1f} hrs)")
print(f"  Est. TOTAL (5 seeds): ~{EST_TOTAL_HRS:.1f} hrs")
print("=" * 60)

seeds      = [0]
all_results = []

for seed in seeds:

    print(f"\n{'='*50}")
    print(f"  Seed: {seed}")
    print(f"{'='*50}")

    set_seed(seed)
    seed_start = time.time()

    # ── Data ──────────────────────────────────────────────────────────────────
    train_dataset = torchvision.datasets.CIFAR10(
        root="./data", train=True, download=True, transform=ssl_transform)

    supervised_train_dataset = torchvision.datasets.CIFAR10(
        root="./data", train=True, download=False, transform=supervised_transform)

    test_dataset = torchvision.datasets.CIFAR10(
        root="./data", train=False, download=False, transform=supervised_transform)

    # FIXED: pin_memory + persistent_workers on ALL loaders
    _loader_kw = dict(
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=True,    # avoid worker respawn between epochs
    )

    train_loader = DataLoader(
        train_dataset, shuffle=True, **_loader_kw)

    supervised_train_loader = DataLoader(
        supervised_train_dataset, shuffle=True, **_loader_kw)

    supervised_test_loader = DataLoader(
        test_dataset, shuffle=False, **_loader_kw)

    # =====================================================
    # BYOL TRAINING
    # =====================================================
    print(f"\n[BYOL] Starting {NUM_EPOCHS}-epoch training")
    byol_model = BYOL().to(device)

    # NEW: torch.compile — fuses ops, significant throughput boost
    try:
        byol_model.online_encoder = torch.compile(
            byol_model.online_encoder, mode="reduce-overhead")
        byol_model.target_encoder = torch.compile(
            byol_model.target_encoder, mode="reduce-overhead")
        print("[BYOL] torch.compile enabled (warm-up on first batch)")
    except Exception:
        print("[BYOL] torch.compile unavailable, skipping")

    optimizer = torch.optim.AdamW(                 # NEW: AdamW > Adam for ViT
        byol_model.parameters(), lr=3e-4, weight_decay=1e-4)

    # NEW: cosine LR schedule — improves final representation quality
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

    scaler = GradScaler("cuda")                    # FIXED: updated API

    byol_epoch_times = []

    for epoch in range(NUM_EPOCHS):
        epoch_start = time.time()
        byol_model.train()
        running_loss = 0.0

        for batch_idx, ((x1, x2), _) in enumerate(train_loader):
            x1 = x1.to(device, non_blocking=True)  # FIXED: non_blocking
            x2 = x2.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)   # FIXED: faster zero_grad

            with autocast("cuda"):                  # FIXED: updated API
                loss = byol_model(x1, x2)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(         # NEW: gradient clipping
                byol_model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item()

        # FIXED: EMA once per epoch, not per batch (~5% speed gain)
        byol_model.update_target()
        scheduler.step()

        elapsed = time.time() - epoch_start
        byol_epoch_times.append(elapsed)

        # ETA
        avg_t   = np.mean(byol_epoch_times)
        remaining = avg_t * (NUM_EPOCHS - epoch - 1)
        print(f"  [BYOL] Epoch {epoch+1:3d}/{NUM_EPOCHS} | "
              f"Loss: {running_loss/len(train_loader):.4f} | "
              f"Time: {elapsed:.1f}s | "
              f"ETA: {remaining/60:.1f} min")

    byol_encoder = byol_model.online_encoder

    # =====================================================
    # SIMCLR TRAINING
    # =====================================================
    print(f"\n[SimCLR] Starting {NUM_EPOCHS}-epoch training")
    simclr_model = SimCLR().to(device)

    try:
        simclr_model.encoder = torch.compile(
            simclr_model.encoder, mode="reduce-overhead")
        print("[SimCLR] torch.compile enabled")
    except Exception:
        print("[SimCLR] torch.compile unavailable, skipping")

    optimizer = torch.optim.AdamW(
        simclr_model.parameters(), lr=3e-4, weight_decay=1e-4)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

    scaler = GradScaler("cuda")

    simclr_epoch_times = []

    for epoch in range(NUM_EPOCHS):
        epoch_start = time.time()
        simclr_model.train()
        running_loss = 0.0

        for batch_idx, ((x1, x2), _) in enumerate(train_loader):
            x1 = x1.to(device, non_blocking=True)
            x2 = x2.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with autocast("cuda"):
                z1   = simclr_model(x1)
                z2   = simclr_model(x2)
                loss = nt_xent_loss(z1, z2, temperature=TEMPERATURE)  # FIXED

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                simclr_model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item()

        scheduler.step()

        elapsed = time.time() - epoch_start
        simclr_epoch_times.append(elapsed)

        avg_t     = np.mean(simclr_epoch_times)
        remaining = avg_t * (NUM_EPOCHS - epoch - 1)
        print(f"  [SimCLR] Epoch {epoch+1:3d}/{NUM_EPOCHS} | "
              f"Loss: {running_loss/len(train_loader):.4f} | "
              f"Time: {elapsed:.1f}s | "
              f"ETA: {remaining/60:.1f} min")

    simclr_encoder = simclr_model.encoder


  Epochs per model  : 100
  Batch size        : 256
  Seeds             : 5
  Est. BYOL/epoch   : ~54 s  (was 90 s)
  Est. SimCLR/epoch : ~81 s  (was 135 s)
  Est. per seed     : ~225 min  (~3.8 hrs)
  Est. TOTAL (5 seeds): ~18.8 hrs

  Seed: 0


100%|██████████| 170M/170M [00:02<00:00, 81.2MB/s] 



[BYOL] Starting 100-epoch training
[BYOL] torch.compile enabled (warm-up on first batch)


W0307 06:21:43.770000 55 torch/_inductor/utils.py:1558] [0/0] Not enough SMs to use max_autotune_gemm mode


  [BYOL] Epoch   1/100 | Loss: 0.9010 | Time: 206.9s | ETA: 341.4 min
  [BYOL] Epoch   2/100 | Loss: 0.8433 | Time: 96.4s | ETA: 247.7 min
  [BYOL] Epoch   3/100 | Loss: 0.8366 | Time: 97.4s | ETA: 215.9 min
  [BYOL] Epoch   4/100 | Loss: 0.8295 | Time: 97.1s | ETA: 199.1 min
  [BYOL] Epoch   5/100 | Loss: 0.8266 | Time: 96.9s | ETA: 188.3 min
  [BYOL] Epoch   6/100 | Loss: 0.8210 | Time: 96.7s | ETA: 180.5 min
  [BYOL] Epoch   7/100 | Loss: 0.8202 | Time: 96.4s | ETA: 174.4 min
  [BYOL] Epoch   8/100 | Loss: 0.8170 | Time: 96.2s | ETA: 169.4 min
  [BYOL] Epoch   9/100 | Loss: 0.8144 | Time: 97.5s | ETA: 165.4 min
  [BYOL] Epoch  10/100 | Loss: 0.8124 | Time: 97.8s | ETA: 161.9 min
  [BYOL] Epoch  11/100 | Loss: 0.8088 | Time: 96.3s | ETA: 158.5 min
  [BYOL] Epoch  12/100 | Loss: 0.8059 | Time: 96.4s | ETA: 155.5 min
  [BYOL] Epoch  13/100 | Loss: 0.8051 | Time: 96.7s | ETA: 152.7 min
  [BYOL] Epoch  14/100 | Loss: 0.8019 | Time: 97.5s | ETA: 150.1 min
  [BYOL] Epoch  15/100 | Loss: 0.

In [12]:
# =====================================================
# EVALUATION
# =====================================================
import os

print("\n[Eval] Running linear probes...")
byol_acc   = train_probe(byol_encoder,   supervised_train_loader, supervised_test_loader)
simclr_acc = train_probe(simclr_encoder, supervised_train_loader, supervised_test_loader)

print("[Eval] Computing geometry metrics...")
byol_jac,   byol_aug,   byol_noise   = compute_metrics(byol_encoder,   supervised_test_loader)
simclr_jac, simclr_aug, simclr_noise = compute_metrics(simclr_encoder, supervised_test_loader)

seed_time = (time.time() - seed_start) / 60
print(f"\n  Seed {seed} finished in {seed_time:.2f} min")
print(f"  BYOL   → Acc: {byol_acc:.4f}  | Jac: {byol_jac:.4f}")
print(f"  SimCLR → Acc: {simclr_acc:.4f}  | Jac: {simclr_jac:.4f}")

SAVE_PATH = "/kaggle/working/cifar10_vit_results.csv"

seed_row = {
    "Seed"              : seed,
    "BYOL_Acc"          : byol_acc,
    "SimCLR_Acc"        : simclr_acc,
    "BYOL_Jacobian"     : byol_jac,
    "SimCLR_Jacobian"   : simclr_jac,
    "BYOL_Aug"          : byol_aug,
    "SimCLR_Aug"        : simclr_aug,
    "BYOL_Noise"        : byol_noise,
    "SimCLR_Noise"      : simclr_noise,
    "Seed_Time_Min"     : seed_time,
    "BYOL_AvgEpochSec"  : np.mean(byol_epoch_times),
    "SimCLR_AvgEpochSec": np.mean(simclr_epoch_times),
}

if os.path.exists(SAVE_PATH):
    results_df = pd.read_csv(SAVE_PATH)
    if seed in results_df["Seed"].values:
        for k, v in seed_row.items():
            results_df.loc[results_df["Seed"] == seed, k] = v
        print(f"  Updated existing row for seed {seed}")
    else:
        results_df = pd.concat([results_df, pd.DataFrame([seed_row])], ignore_index=True)
        print(f"  Appended seed {seed}")
else:
    results_df = pd.DataFrame([seed_row])
    print(f"  Created new results file for seed {seed}")

results_df.to_csv(SAVE_PATH, index=False)
print(f"  Saved → {SAVE_PATH}")


[Eval] Running linear probes...
[Eval] Computing geometry metrics...

  Seed 0 finished in 332.91 min
  BYOL   → Acc: 0.4906  | Jac: 1.4900
  SimCLR → Acc: 0.7660  | Jac: 5.6087
  Created new results file for seed 0
  Saved → /kaggle/working/cifar10_vit_results.csv


In [14]:
LAMBDA_JAC   = 1e-3
TEMPERATURE  = 1.0                                  # τ = 1.0 as determined
NUM_EPOCHS   = 100
BATCH_SIZE   = 256
NUM_WORKERS  = 4

print("=" * 60)
print(f"  [JacReg] SimCLR + Jacobian Reg  |  Seed: {seed}")
print(f"  lambda_jac = {LAMBDA_JAC}  |  tau = {TEMPERATURE}")
print("=" * 60)

set_seed(seed)
jacreg_start = time.time()

# ── Data (CIFAR-10, same seed as BYOL/SimCLR cells) ──────────────────────────
train_dataset = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=ssl_transform)

supervised_train_dataset = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=False, transform=supervised_transform)

test_dataset = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=False, transform=supervised_transform)

_loader_kw = dict(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
)

train_loader            = DataLoader(train_dataset,            shuffle=True,  **_loader_kw)
supervised_train_loader = DataLoader(supervised_train_dataset, shuffle=True,  **_loader_kw)
supervised_test_loader  = DataLoader(test_dataset,             shuffle=False, **_loader_kw)

# ── Model ─────────────────────────────────────────────────────────────────────
jacreg_model = SimCLR().to(device)


optimizer = torch.optim.AdamW(
    jacreg_model.parameters(), lr=3e-4, weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

scaler = GradScaler("cuda")

jacreg_epoch_times = []

for epoch in range(NUM_EPOCHS):
    epoch_start  = time.time()
    jacreg_model.train()
    running_loss = 0.0
    running_ssl  = 0.0
    running_jac  = 0.0

    for batch_idx, ((x1, x2), _) in enumerate(train_loader):

        # requires_grad needed for Jacobian — only on x1
        x1 = x1.to(device, non_blocking=True).requires_grad_(True)
        x2 = x2.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast("cuda"):
            z1 = jacreg_model(x1)
            z2 = jacreg_model(x2)
            loss_ssl = nt_xent_loss(z1, z2, temperature=TEMPERATURE)

        # FIXED: compute Jac reg every other batch (same as original)
        # but keep it inside autocast for speed, and use retain_graph correctly
        if batch_idx % 2 == 0:
            with autocast("cuda"):
                features = jacreg_model.encoder(x1)

            v  = torch.randn_like(features)
            Jv = torch.autograd.grad(
                outputs=features,
                inputs=x1,
                grad_outputs=v,
                retain_graph=False,
                create_graph=False,         # NEW: no 2nd-order graph, faster
            )[0]
            # FIXED: detach before .item() to avoid holding graph
            jac_reg = Jv.pow(2).sum(dim=[1, 2, 3]).mean()
        else:
            jac_reg = torch.tensor(0.0, device=device)

        loss = loss_ssl + LAMBDA_JAC * jac_reg

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(
            jacreg_model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        running_ssl  += loss_ssl.item()
        running_jac  += jac_reg.item() if isinstance(jac_reg, torch.Tensor) else 0.0

    scheduler.step()

    elapsed = time.time() - epoch_start
    jacreg_epoch_times.append(elapsed)
    avg_t     = np.mean(jacreg_epoch_times)
    remaining = avg_t * (NUM_EPOCHS - epoch - 1)
    n_batches = len(train_loader)

    print(f"  [JacReg] Epoch {epoch+1:3d}/{NUM_EPOCHS} | "
          f"Loss: {running_loss/n_batches:.4f}  "
          f"(SSL: {running_ssl/n_batches:.4f}  "
          f"Jac: {running_jac/n_batches:.4f}) | "
          f"Time: {elapsed:.1f}s | "
          f"ETA: {remaining/60:.1f} min")

jacreg_encoder = jacreg_model.encoder

# ── Evaluate ──────────────────────────────────────────────────────────────────
print("\n[JacReg] Running linear probe...")
jacreg_acc = train_probe(
    jacreg_encoder, supervised_train_loader, supervised_test_loader)

print("[JacReg] Computing geometry metrics...")
jacreg_jac, jacreg_aug, jacreg_noise = compute_metrics(
    jacreg_encoder, supervised_test_loader)

jacreg_time = (time.time() - jacreg_start) / 60

print(f"\n  [JacReg] Seed {seed} done in {jacreg_time:.2f} min")
print(f"  JacReg → Acc: {jacreg_acc:.4f}  | Jac: {jacreg_jac:.4f}")


# =========================================================
# CELL 2 — Final Results  (all 3 models, 1 seed)
# =========================================================
#
# Reads BYOL + SimCLR results from all_results (built in prior cells),
# appends JacReg, saves everything to /kaggle/working/
# ─────────────────────────────────────────────────────────

import os

# ── Build unified row for this seed ───────────────────────────────────────────
seed_row = {
    "Seed"                  : seed,

    # Accuracy
    "BYOL_Acc"              : byol_acc,
    "SimCLR_Acc"            : simclr_acc,
    "JacReg_Acc"            : jacreg_acc,

    # Jacobian sensitivity
    "BYOL_Jacobian"         : byol_jac,
    "SimCLR_Jacobian"       : simclr_jac,
    "JacReg_Jacobian"       : jacreg_jac,

    # Augmentation invariance
    "BYOL_Aug"              : byol_aug,
    "SimCLR_Aug"            : simclr_aug,
    "JacReg_Aug"            : jacreg_aug,

    # Noise robustness
    "BYOL_Noise"            : byol_noise,
    "SimCLR_Noise"          : simclr_noise,
    "JacReg_Noise"          : jacreg_noise,

    # Timing
    "BYOL_Time_Min"         : sum(byol_epoch_times) / 60,
    "SimCLR_Time_Min"       : sum(simclr_epoch_times) / 60,
    "JacReg_Time_Min"       : jacreg_time,

    "BYOL_AvgEpochSec"      : np.mean(byol_epoch_times),
    "SimCLR_AvgEpochSec"    : np.mean(simclr_epoch_times),
    "JacReg_AvgEpochSec"    : np.mean(jacreg_epoch_times),
}

# ── Merge with any previously saved seeds ─────────────────────────────────────
SAVE_PATH = "/kaggle/working/cifar10_results_progress.csv"

if os.path.exists(SAVE_PATH):
    results_df = pd.read_csv(SAVE_PATH)
    # overwrite row if seed already exists, else append
    if seed in results_df["Seed"].values:
        for k, v in seed_row.items():
            results_df.loc[results_df["Seed"] == seed, k] = v
        print(f"  Updated existing row for seed {seed}")
    else:
        results_df = pd.concat(
            [results_df, pd.DataFrame([seed_row])], ignore_index=True)
        print(f"  Appended new row for seed {seed}")
else:
    results_df = pd.DataFrame([seed_row])
    print(f"  Created new results file for seed {seed}")

results_df.to_csv(SAVE_PATH, index=False)
print(f"  Saved → {SAVE_PATH}")

# ── Pretty-print side-by-side comparison ──────────────────────────────────────
METRICS = [
    ("Accuracy",              "BYOL_Acc",       "SimCLR_Acc",       "JacReg_Acc"),
    ("Jacobian Sensitivity",  "BYOL_Jacobian",  "SimCLR_Jacobian",  "JacReg_Jacobian"),
    ("Aug Invariance",        "BYOL_Aug",       "SimCLR_Aug",       "JacReg_Aug"),
    ("Noise Robustness",      "BYOL_Noise",     "SimCLR_Noise",     "JacReg_Noise"),
    ("Train Time (min)",      "BYOL_Time_Min",  "SimCLR_Time_Min",  "JacReg_Time_Min"),
    ("Avg Epoch (sec)",       "BYOL_AvgEpochSec","SimCLR_AvgEpochSec","JacReg_AvgEpochSec"),
]

print("\n" + "=" * 70)
print(f"  SEED {seed} RESULTS  —  BYOL vs SimCLR vs SimCLR+JacReg  (CIFAR-10)")
print("=" * 70)
print(f"  {'Metric':<26}  {'BYOL':>10}  {'SimCLR':>10}  {'JacReg':>10}")
print("  " + "-" * 62)
for label, b_col, s_col, j_col in METRICS:
    b = seed_row[b_col]
    s = seed_row[s_col]
    j = seed_row[j_col]
    print(f"  {label:<26}  {b:>10.4f}  {s:>10.4f}  {j:>10.4f}")
print("=" * 70)

# ── If multiple seeds are done, also print aggregate stats ────────────────────
if len(results_df) > 1:
    print(f"\n  AGGREGATE  ({len(results_df)} seeds completed)")
    print("=" * 70)
    print(f"  {'Metric':<26}  {'BYOL':>10}  {'SimCLR':>10}  {'JacReg':>10}")
    print("  " + "-" * 62)
    for label, b_col, s_col, j_col in METRICS:
        bm = results_df[b_col].mean(); bs = results_df[b_col].std()
        sm = results_df[s_col].mean(); ss = results_df[s_col].std()
        jm = results_df[j_col].mean(); js = results_df[j_col].std()
        print(f"  {label:<26}  "
              f"{bm:>6.4f}±{bs:.3f}  "
              f"{sm:>6.4f}±{ss:.3f}  "
              f"{jm:>6.4f}±{js:.3f}")
    print("=" * 70)

print(f"\n  Seeds completed so far: {sorted(results_df['Seed'].tolist())}")
print(f"  Progress CSV: {SAVE_PATH}")


  [JacReg] SimCLR + Jacobian Reg  |  Seed: 0
  lambda_jac = 0.001  |  tau = 1.0
  [JacReg] Epoch   1/100 | Loss: 5.7034  (SSL: 5.7004  Jac: 2.9610) | Time: 99.9s | ETA: 164.9 min
  [JacReg] Epoch   2/100 | Loss: 5.6188  (SSL: 5.6176  Jac: 1.1585) | Time: 100.5s | ETA: 163.7 min
  [JacReg] Epoch   3/100 | Loss: 5.5967  (SSL: 5.5956  Jac: 1.0997) | Time: 101.2s | ETA: 162.6 min
  [JacReg] Epoch   4/100 | Loss: 5.5802  (SSL: 5.5790  Jac: 1.2936) | Time: 101.3s | ETA: 161.2 min
  [JacReg] Epoch   5/100 | Loss: 5.5694  (SSL: 5.5680  Jac: 1.4297) | Time: 101.1s | ETA: 159.6 min
  [JacReg] Epoch   6/100 | Loss: 5.5594  (SSL: 5.5579  Jac: 1.5719) | Time: 99.3s | ETA: 157.5 min
  [JacReg] Epoch   7/100 | Loss: 5.5508  (SSL: 5.5492  Jac: 1.5651) | Time: 100.1s | ETA: 155.8 min
  [JacReg] Epoch   8/100 | Loss: 5.5436  (SSL: 5.5419  Jac: 1.6624) | Time: 99.3s | ETA: 153.9 min
  [JacReg] Epoch   9/100 | Loss: 5.5355  (SSL: 5.5336  Jac: 1.9274) | Time: 99.4s | ETA: 152.0 min
  [JacReg] Epoch  10/100